[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/42_tool_use_solution.ipynb)

#  Solution: Tool Use (Function Calling)

Reference solution for tool-call parsing and execution.

```text
1. Try to parse response as JSON
2. Extract "tool" and "args" keys
3. Look up tool in tools dict, call with unpacked args
4. Return (tool_name, result) or None
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import json

In [ ]:
#  SOLUTION

def execute_tool_call(response_text, tools):
    """Parse and execute tool calls from model response.
    
    Inspired by OpenAI function-calling / Anthropic tool-use format.
    Parses JSON-style: {"tool": "name", "args": {...}}
    """
    try:
        data = json.loads(response_text)
        if isinstance(data, dict) and "tool" in data and "args" in data:
            tool_name = data["tool"]
            if tool_name in tools:
                args = data["args"]
                result = tools[tool_name](**args)
                return (tool_name, result)
    except (json.JSONDecodeError, TypeError, KeyError):
        pass
    return None

In [ ]:
#  Verify
tools = {"add": lambda a, b: a + b, "greet": lambda name: f"Hello {name}!"}
print(execute_tool_call('{"tool": "add", "args": {"a": 1, "b": 2}}', tools))
print(execute_tool_call('{"tool": "greet", "args": {"name": "World"}}', tools))
print(execute_tool_call('Just a normal reply.', tools))

In [ ]:
# Run judge
from torch_judge import check
check("tool_use")